In [21]:
# 翻译数据集

import threading
import time
import pandas
import requests
import uuid

import tqdm

In [22]:
datas = pandas.read_parquet('datas.parquet').sample(10000)

In [23]:
tags = {}
locks = []
lk = threading.Lock()

def fanyi(text: str) -> str:
    url = "https://api.deepseek.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer sk-be7adce9ea1b4d93bf8ccc1a8e879c8c",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "deepseek-chat",
        "messages": [
            {
                "role": "user",
                "content": f"依照游戏minecraft语境 请将以下英文翻译成中文，只返回翻译结果：{text}"
            }
        ],
        "stream": False
    }

    try:
        resp = requests.post(url, json=payload, headers=headers, timeout=30)
        resp.raise_for_status()
        res_data = resp.json()
        return res_data["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"翻译失败：{str(e)}"


def fanyi_tag(text):
    with lk:
        lock = uuid.uuid4()
        locks.append(lock)
    url = "https://api.deepseek.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer xxx",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "deepseek-chat",
        "messages": [
            {
                "role": "user",
                "content": f"依照游戏minecraft语境 请将以下tag翻译成中文, 只给我内容本身 不输出其他内容：{text}"
            }
        ],
        "stream": False
    }

    try:
        resp = requests.post(url, json=payload, headers=headers, timeout=30)
        resp.raise_for_status()
        res_data = resp.json()
        tags[text] = res_data["choices"][0]["message"]["content"].strip()
        with lk:
            locks.remove(lock)
        return res_data["choices"][0]["message"]["content"].strip()
    except Exception as e:
        with lk:
            locks.remove(lock)
        return f"翻译失败：{str(e)}"


def func(row):
    row['question'] = fanyi(row['question'])
    row['answer'] = fanyi(row['answer'])
    if str(row['source']) not in tags:
        row['source'] = fanyi_tag(str(row['source']))
    else:
        row['source'] = tags[str(row['source'])]
    return row

In [24]:
def f(idx, row):
    # 执行函数处理当前行
    new_row = func(row)
    # 用处理后的行 替换原 DataFrame 对应行
    with lk:
        datas.loc[idx] = new_row

for idx, row in tqdm.tqdm(datas.iterrows()):
    while len(locks) > 50: time.sleep(1)
    threading.Thread(target=f, args=(idx, row, )).start()

while locks:  # 等待结束
    print(len(locks))
    time.sleep(1)

10000it [01:47, 92.85it/s]


44
48
35
19
7


In [25]:
print(tags)
datas.to_csv('datas.csv')

{'Potion_Of_Harming_II': '伤害药水 II', 'End_Dimension': '末地维度', 'Dirt': '泥土', 'Diamond_Armor': '钻石盔甲', 'Birch_Sapling': '白桦树苗', 'Logic_Circuit': '逻辑电路', 'Amethyst_Trim_Iron_Helmet': '紫晶纹饰铁头盔', 'Pocket_Edition_V0.1.1': 'Pocket_Edition_V0.1.1', 'Bedrock_Edition_V1.2.2': '基岩版V1.2.2', 'Java_Edition_Removed_Features': 'Java版已移除特性', 'Gray_Wool': '灰色羊毛', 'Amethyst_Trim_Leather_Pants': '紫水晶镶边皮革裤', 'Argument_Types': '参数类型', 'Huge_Crimson_Fungi': '巨型绯红菌', 'Bamboo_Raft_With_Chest': '竹筏_带箱子', 'Pocket_Edition_1.0.6': '携带版1.0.6', 'Dead_Fire_Coral_Fan': '死掉的火焰珊瑚扇', 'Dragon_Wall_Head': '龙墙头颅', 'Zombie_Pigman_Spawn_Egg': '僵尸猪人生成蛋', 'Potions_Of_Invisibility': '隐形药水', 'Density_Function': '密度函数', 'Display_Entity': '展示实体', 'Campfire': '篝火', 'Java_Edition_1.12.2/Development_Versions': 'Java版1.12.2/开发版本', 'Bedrock_Edition_1.2.13/Development_Versions': '基岩版1.2.13/开发版本', 'Teams': '队伍', 'Sea_Grass': '海草', 'Gray_Shulker_Box': '灰色潜影盒', 'Tutorials/Village_Mechanics': '教程/村庄机制', 'Golden_Horse_Armor': '金马铠', 'Double_Ch